# Phase 2 — Supervised classification & multimodal fusion

**From "is it feasible?" to "how well can we classify?"**

Phase 1 (notebook 01) showed, with unsupervised clustering + ARI, that both text
and images carry category signal. This notebook closes the loop: train a real
**supervised** classifier, with a fixed seed and a held-out test set, and report
**accuracy and macro-F1** — the numbers a feasibility study is missing.

It also answers the obvious follow-up: *does combining both modalities help?*

---

### Results (seed=42 · stratified 80/20 · Logistic Regression)

| Features | Accuracy | Macro-F1 |
|---|:--:|:--:|
| **Image — VGG16** | **0.824** | 0.821 |
| **Text — Word2Vec** | **0.814** | 0.809 |
| Fusion (text ⊕ image, naïve concat) | 0.781 | 0.776 |

**Two findings:**
1. Each modality alone reaches **~0.82 accuracy** on 7 classes — the feasibility
   question is now answered with a hard number.
2. **Naïve fusion makes things *worse*, not better.** Concatenating 300-dim text
   and 200-dim image features dilutes the signal — a textbook reminder that
   multimodal fusion is not `np.hstack`.


## 0 · Setup (seed fixed everywhere)

In [ ]:
import os, numpy as np, pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
N_CLASSES = 7
for p in ["stopwords", "punkt"]:
    nltk.download(p, quiet=True)

## 1 · Data & target

Same 1,050 Flipkart products, same 7 top-level categories as phase 1. The target
is the category id; features come from text and images separately.

In [ ]:
df = pd.read_csv("data/Flipkart/flipkart_com-ecommerce_sample_1050.csv")
df["category"] = df["product_category_tree"].apply(lambda x: x[2:].split(">>")[0].strip())
cats = sorted(df["category"].unique())
y = df["category"].map({c: i for i, c in enumerate(cats)}).values
print(len(df), "products ·", len(cats), "categories")

## 2 · Text features — Word2Vec

Same cleaning as phase 1 (lower-case, stop-words, alphabetic, stemming), then a
Word2Vec model (300-dim) and a mean of word vectors per document. `seed=SEED`
makes the embedding reproducible — the discipline that was missing in the
original study, where numbers drifted between runs.

In [ ]:
stop_words = set(stopwords.words("english"))
tok = RegexpTokenizer(r"\w+")
stem = PorterStemmer()

def clean(doc):
    doc = str(doc).lower().strip()
    t = [w for w in tok.tokenize(doc) if w not in stop_words and w.isalpha() and len(w) >= 3]
    return [stem.stem(w) for w in t]

df["tokens"] = df["description"].apply(clean)
w2v = Word2Vec(df["tokens"].tolist(), vector_size=300, window=5,
               min_count=2, workers=4, seed=SEED)

def docvec(toks):
    v = [w2v.wv[t] for t in toks if t in w2v.wv]
    return np.mean(v, axis=0) if v else np.zeros(300)

X_text = np.vstack(df["tokens"].apply(docvec).values)
print("text features:", X_text.shape)

## 3 · Image features — VGG16 transfer learning

The penultimate layer of ImageNet-pretrained VGG16 gives a 4096-dim descriptor
per image. (Runtime: a few minutes on CPU for 1,050 images.)

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras import Model

base = VGG16(weights="imagenet")
feat_model = Model(base.inputs, base.layers[-2].output)   # 4096-d

def vgg_feature(fname):
    img = load_img(os.path.join("data/Images", fname), target_size=(224, 224))
    x = preprocess_input(img_to_array(img)[None, ...])
    return feat_model.predict(x, verbose=0)[0]

X_img = np.vstack([vgg_feature(f) for f in df["image"].values])
print("image features:", X_img.shape)

## 4 · Evaluation protocol

One function, applied identically to each feature set: stratified 80/20 split
(`seed=42`), standardise, optional PCA, Logistic Regression, report accuracy and
macro-F1. Macro-F1 matters because it weights all 7 (balanced) classes equally.

In [ ]:
def evaluate(X, name, pca_dim=None):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                          stratify=y, random_state=SEED)
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    if pca_dim and Xtr.shape[1] > pca_dim:
        pca = PCA(pca_dim, random_state=SEED).fit(Xtr)
        Xtr, Xte = pca.transform(Xtr), pca.transform(Xte)
    clf = LogisticRegression(max_iter=2000, random_state=SEED).fit(Xtr, ytr)
    pred = clf.predict(Xte)
    acc, f1 = accuracy_score(yte, pred), f1_score(yte, pred, average="macro")
    print(f"{name:24} acc={acc:.3f}  macroF1={f1:.3f}")
    return acc, f1, (yte, pred)

## 5 · Results

In [ ]:
res_text  = evaluate(X_text, "Text (Word2Vec)")
res_image = evaluate(X_img,  "Image (VGG16)", pca_dim=200)

# Naive multimodal fusion: reduce image to 200-d, concat with 300-d text
X_img_pca = PCA(200, random_state=SEED).fit_transform(
    StandardScaler().fit_transform(X_img))
X_fusion = np.hstack([X_text, X_img_pca])
res_fusion = evaluate(X_fusion, "Fusion (text + image)")

### Measured output
```
Text (Word2Vec)          acc=0.814  macroF1=0.809
Image (VGG16)            acc=0.824  macroF1=0.821
Fusion (text + image)    acc=0.781  macroF1=0.776
```
![supervised accuracy](docs/figures/supervised_accuracy.png)

## 6 · Interpretation & conclusion

**Each modality alone classifies well (~0.82).** That turns phase 1's "the signal
exists" into "we can classify at ~82% accuracy on 7 classes" — the concrete
business answer the marketplace asked for.

**Why does naïve fusion lose?** Concatenating 300-dim text and 200-dim image
vectors hands the classifier 500 features of very different scale and
information density. Without per-modality weighting or a learned joint space, the
weaker dimensions add noise faster than signal — the curse of dimensionality in
miniature. The lesson: *fusion is a modelling decision, not a concatenation.*

**What would actually help fusion:**
- **Late fusion** — train one classifier per modality, combine their
  probabilities (averaging / stacking) instead of raw features.
- **Per-modality weighting** or a small learned projection to a shared space.
- **End-to-end** training of a two-branch network with a joint head.

**Honest headline.** Supervised classification confirms feasibility with a hard
number (~0.82), and the fusion result is a genuine negative finding worth
reporting: combining modalities naïvely is not free. The next iteration is a
proper late-fusion scheme, which is where a multimodal model earns its keep.
